# `counter` demo — causal attribution on agent traces

Two layers:

1. **Synthetic SCM sanity check** — generate traces from a *known* causal process; show that `intervene()` recovers the true effect within tolerance, and the bootstrap CI covers it.
2. **Mixed-identifiability case study** — call `intervene()` for queries that exercise all three identifiability paths (`identified`, `bounded`, `unidentified`) and render each result with its concrete `reason` / `next_step` / `bounds`.

The third layer — a real-agent trace case study with hand-labeled root cause — lands once `§14.1` clears.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

from bench.synthetic import HEADLINE_TRUE_EFFECT, generate_traces
from counter import attribute_failure, build_dag, fit_outcome_model, intervene
from counter.intervene import IdentifiabilityStatus
from counter.schema import Decision, Outcome, Run, Step

print(f"Python {sys.version.split()[0]} · headline true effect (sonnet vs haiku marginal): {HEADLINE_TRUE_EFFECT:.4f}")

## 1. Synthetic SCM sanity check

We generate 500 traces from a known structural causal model with three randomized decision types (`tool_choice`, `model_choice`, `retry_policy`). The headline intervention is `model_choice`: sonnet versus haiku, marginalized over the other arms. Its true effect on `P(success)` is exposed as `HEADLINE_TRUE_EFFECT`.

We fit the outcome model, run `intervene()` on each arm, take the difference, and compare to truth.

In [ ]:
runs = [Run.model_validate(t) for t in generate_traces(n=500, seed=42)]
model = fit_outcome_model(runs, n_bootstrap=200, seed=42)

dag = build_dag(runs[0])  # any run carries the same step structure for this SCM

p_sonnet = intervene(dag=dag, model=model, step=2, intervention={"model_choice": "sonnet"})
p_haiku  = intervene(dag=dag, model=model, step=2, intervention={"model_choice": "haiku"})

estimated = p_sonnet.outcome_delta.point - p_haiku.outcome_delta.point
print(f"  estimated effect: {estimated:+.4f}")
print(f"  true effect:      {HEADLINE_TRUE_EFFECT:+.4f}")
print(f"  |diff|:           {abs(estimated - HEADLINE_TRUE_EFFECT):.4f}  (tolerance ±0.05)")
assert abs(estimated - HEADLINE_TRUE_EFFECT) <= 0.05, "SCM-recovery tolerance violated"

## 2. An *identified* result — `tool_choice` with randomized support

The synthetic corpus is uniform-randomized over tool arms, so the back-door criterion is satisfied by construction. We expect `identifiability="identified"` with a non-null `outcome_delta`, an explicit `adjustment_set`, and assumptions naming the strategy.

In [ ]:
identified = intervene(dag=dag, model=model, step=1, intervention={"tool_choice": "run_tests"})

print(f"identifiability : {identified.identifiability.value}")
print(f"point estimate  : {identified.outcome_delta.point:+.4f}")
print(f"95% bootstrap CI: [{identified.outcome_delta.ci_low:+.4f}, {identified.outcome_delta.ci_high:+.4f}]")
print(f"E-value         : {identified.bounds.e_value:.3f}")
print(f"adjustment_set  : {identified.adjustment_set[:5]}{'...' if len(identified.adjustment_set) > 5 else ''}")
print()
print("assumptions:")
for a in identified.assumptions:
    print(f"  - {a}")
assert identified.identifiability == IdentifiabilityStatus.IDENTIFIED
assert any("adjustment" in a for a in identified.assumptions)

## 3. A *bounded* result — `memory_content` requires back-door adjustment

Memory contents are high-dimensional and the synthetic corpus has limited support. The taxonomy declares `memory_content` as `requires-back-door-adjustment`, so `intervene()` returns a *bounded* estimate: it names the adjustment strategy in `assumptions`, populates `bounds.e_value`, and warns the user explicitly.

In [ ]:
# Construct a one-off run with a memory_read step so we can demo the bounded path
# even though the v0 synthetic SCM doesn't include memory reads.
mem_run = Run(
    schema_version="0.1.0",
    run_id="demo-mem-001",
    steps=[
        Step(step_index=0, decisions=[Decision(decision_id="d0", decision_type="plan_step", chosen_action="begin")]),
        Step(step_index=1, decisions=[Decision(decision_id="d1", decision_type="memory_read", chosen_action="recent_5")]),
    ],
    outcome=Outcome(kind="binary", value=False, verifier="pytest"),
)
bounded = intervene(
    dag=build_dag(mem_run),
    model=model,
    step=1,
    intervention={"memory_content": "all"},
)

print(f"identifiability : {bounded.identifiability.value}")
print(f"E-value         : {bounded.bounds.e_value:.3f}  ({bounded.bounds.technique})")
print(f"adjustment_set  : {bounded.adjustment_set}")
print()
print("assumptions:")
for a in bounded.assumptions:
    print(f"  - {a}")
print()
print("warnings:")
for w in bounded.warnings:
    print(f"  ! {w}")
assert bounded.identifiability == IdentifiabilityStatus.BOUNDED
assert bounded.bounds is not None

## 4. An *unidentified* result — `prompt_content` is replay-only

The taxonomy treats prompt-content interventions as `always-replay`: the prompt is high-dimensional, randomization in the corpus does not cover it, and the LLM completion is opaque (design.md D1). `intervene()` returns `unidentified` with a concrete `reason` and `next_step="replay"` — exactly the discipline this library was built for.

In [ ]:
unidentified = intervene(
    dag=dag,
    model=model,
    step=2,  # model_call step in synthetic SCM
    intervention={"prompt_content": "think step by step"},
)

print(f"identifiability : {unidentified.identifiability.value}")
print(f"reason          : {unidentified.reason}")
print(f"next_step       : {unidentified.next_step}")
print()
print("assumptions:")
for a in unidentified.assumptions:
    print(f"  - {a}")
print()
print("warnings:")
for w in unidentified.warnings:
    print(f"  ! {w}")
assert unidentified.identifiability == IdentifiabilityStatus.UNIDENTIFIED
assert unidentified.next_step == "replay"
assert unidentified.reason

## 5. Ranked failure attribution

Pick a failed synthetic run; rank decisions by their estimated causal influence on the outcome. Each entry carries an identifiability label, so callers can filter by epistemic confidence.

In [ ]:
failed_runs = [r for r in runs if r.outcome.value is False]
print(f"failed runs in corpus: {len(failed_runs)} / {len(runs)}")

case = failed_runs[0]
attribution = attribute_failure(dag=build_dag(case), model=model)
top5 = attribution.top_k(5)

print(f"\nfailure attribution for {case.run_id}:")
print(f"{'rank':>4}  {'decision_id':<22}  {'type':<12}  {'action':<14}  {'influence':>9}  identifiability")
print("-" * 88)
for i, e in enumerate(top5, start=1):
    print(
        f"{i:>4}  {e.decision_id:<22}  {e.decision_type:<12}  {e.chosen_action:<14}  "
        f"{e.influence:>+9.4f}  {e.identifiability.value}"
    )
assert len(top5) <= 5
assert all(
    e.identifiability
    in {IdentifiabilityStatus.IDENTIFIED, IdentifiabilityStatus.BOUNDED, IdentifiabilityStatus.UNIDENTIFIED}
    for e in top5
)

## Takeaways

- The synthetic SCM canary recovers the known headline effect within ±0.05 — the schema → DAG → outcome model → identifiability dispatch pipeline is *correct on a known-truth case*.
- Every `intervene()` answer carries either a calibrated point + bootstrap CI + E-value (when identified), a sensitivity bound + adjustment-set name (when bounded), or a concrete `reason` + `next_step="replay"` (when unidentified). No silent Pearl-L3 claims sneak in.
- Failure attribution propagates identifiability labels to per-decision influence scores, so downstream consumers can filter or weight by epistemic confidence rather than treating all rankings as equal.

**Pending §12.3 / §14.1:** the real-agent case study and hand-labeled top-1 root-cause comparison append below this section once the corpus + label land.